In [36]:
import torch

import triton
import triton.language as tl
import os
import json

DEVICE = triton.runtime.driver.active.get_active_torch_device()

def get_config(config_file_path):
    if os.path.exists(config_file_path):
        with open(config_file_path) as f:
            return {int(key): val for key, val in json.load(f).items()}

@triton.jit
def matmul_kernel_3d(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        expert_ids_ptr,
        # The stride variables represent how much to increase the ptr by when moving by 1
        # element in a particular dimension. E.g. `stride_am` is how much to increase `a_ptr`
        # by to get the element one row down (A has M rows).
        stride_am, stride_ak,  #
        stride_be, stride_bk, stride_bn,  #
        stride_cm, stride_cn,
        top_k: tl.constexpr,  #
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,  #
        GROUP_SIZE_M: tl.constexpr,  #
        # ACTIVATION: tl.constexpr  #
):
    """Kernel for computing the matmul C = A x B.
    A has shape (M, K), B has shape (K, N) and C has shape (M, N)
    """
    # -----------------------------------------------------------
    # Map program ids `pid` to the block of C it should compute.
    # This is done in a grouped ordering to promote L2 data reuse.
    # See above `L2 Cache Optimizations` section for details.
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # ----------------------------------------------------------
    # Create pointers for the first blocks of A and B.
    # We will advance this pointer as we move in the K direction
    # and accumulate
    # `a_ptrs` is a block of [BLOCK_SIZE_M, BLOCK_SIZE_K] pointers
    # `b_ptrs` is a block of [BLOCK_SIZE_K, BLOCK_SIZE_N] pointers
    # See above `Pointer Arithmetic` section for details
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    off_experts = tl.load(expert_ids_ptr + pid_m).to(tl.int64)
    a_ptrs = a_ptr + (offs_am[:, None] // top_k * stride_am + offs_k[None, :] * stride_ak)
    # b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)
    b_ptrs = b_ptr + off_experts * stride_be + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    # -----------------------------------------------------------
    # Iterate to compute a block of the C matrix.
    # We accumulate into a `[BLOCK_SIZE_M, BLOCK_SIZE_N]` block
    # of fp32 values for higher accuracy.
    # `accumulator` will be converted back to fp16 after the loop.
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        # Load the next block of A and B, generate a mask by checking the K dimension.
        # If it is out of bounds, set it to 0.
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        # We accumulate along the K dimension.
        accumulator = tl.dot(a, b, accumulator)
        # Advance the ptrs to the next K block.
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    # You can fuse arbitrary activation functions here
    # while the accumulator is still in FP32!
    # if ACTIVATION == "leaky_relu":
    #     accumulator = leaky_relu(accumulator)
    c = accumulator.to(tl.float16)

    # -----------------------------------------------------------
    # Write back the block of the output matrix C with masks.
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

@triton.jit
def matmul_kernel_2d(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        # The stride variables represent how much to increase the ptr by when moving by 1
        # element in a particular dimension. E.g. `stride_am` is how much to increase `a_ptr`
        # by to get the element one row down (A has M rows).
        stride_am, stride_ak,  #
        stride_bk, stride_bn,  #
        stride_cm, stride_cn,
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,  #
        GROUP_SIZE_M: tl.constexpr,  #
):
    """Kernel for computing the matmul C = A x B.
    A has shape (M, K), B has shape (K, N) and C has shape (M, N)
    """
    # -----------------------------------------------------------
    # Map program ids `pid` to the block of C it should compute.
    # This is done in a grouped ordering to promote L2 data reuse.
    # See above `L2 Cache Optimizations` section for details.
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # -----------------------------------------------------------
    # Add some integer bound assumptions.
    # This helps to guide integer analysis in the backend to optimize
    # load/store offset address calculation
    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    # ----------------------------------------------------------
    # Create pointers for the first blocks of A and B.
    # We will advance this pointer as we move in the K direction
    # and accumulate
    # `a_ptrs` is a block of [BLOCK_SIZE_M, BLOCK_SIZE_K] pointers
    # `b_ptrs` is a block of [BLOCK_SIZE_K, BLOCK_SIZE_N] pointers
    # See above `Pointer Arithmetic` section for details
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    # -----------------------------------------------------------
    # Iterate to compute a block of the C matrix.
    # We accumulate into a `[BLOCK_SIZE_M, BLOCK_SIZE_N]` block
    # of fp32 values for higher accuracy.
    # `accumulator` will be converted back to fp16 after the loop.
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        # Load the next block of A and B, generate a mask by checking the K dimension.
        # If it is out of bounds, set it to 0.
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        # We accumulate along the K dimension.
        accumulator = tl.dot(a, b, accumulator)
        # Advance the ptrs to the next K block.
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    c = accumulator.to(tl.float16)

    # -----------------------------------------------------------
    # Write back the block of the output matrix C with masks.
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)
    # TODO: Tryme
    # tl.store(c_ptr, tl.sum(a_ptrs[:, None] * b_ptrs, 0))

def matmul(a, b, num_experts, config):
    if num_experts > 1:
        top_k_num = 1
        # Check constraints.
        assert a.shape[1] == b.shape[2], "Incompatible dimensions"
        assert a.is_contiguous(), "Matrix A must be contiguous"
        assert b.shape[0] == num_experts, "Number of experts does not match"
        M, K = a.shape
        num_experts, N, K = b.shape
        # Allocates output.
        c = torch.empty((M, top_k_num, N), device=a.device, dtype=torch.float16)
        # 1D launch kernel where each block gets its own program.
        grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
        activated_experts = min(M, num_experts)
        expert_ids = torch.arange(activated_experts, device=a.device, dtype=torch.int32).view(1, -1)

        matmul_kernel_3d[grid](
            a, b, c,  #
            M, N, K,  #
            expert_ids, 
            a.stride(0), a.stride(1),  #
            b.stride(0), b.stride(2), b.stride(1),  #
            c.stride(1), c.stride(2),  #
            top_k=top_k_num,  #
            # ACTIVATION=activation  #
            **config
        )
    else:
        # Check constraints.
        assert a.shape[1] == b.shape[0], "Incompatible dimensions"
        assert a.is_contiguous(), "Matrix A must be contiguous"
        M, K = a.shape
        K, N = b.shape
        # Allocates output.
        c = torch.empty((M, N), device=a.device, dtype=torch.float16)
        # 1D launch kernel where each block gets its own program.
        grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
        matmul_kernel_2d[grid](
            a, b, c,  #
            M, N, K,  #
            a.stride(0), a.stride(1),  #
            b.stride(0), b.stride(1),  #
            c.stride(0), c.stride(1),  #
            **config
        )
    return c

@triton.jit
def grouped_matmul_kernel(
    # device tensor of matrices pointers
    group_a_ptrs,
    group_b_ptrs,
    group_c_ptrs,
    # device tensor of gemm sizes. its shape is [group_size, 3]
    # dim 0 is group_size, dim 1 is the values of <M, N, K> of each gemm
    group_gemm_sizes,
    # device tensor of leading dimension sizes. its shape is [group_size, 3]
    # dim 0 is group_size, dim 1 is the values of <lda, ldb, ldc> of each gemm
    g_lds,
    # number of gemms
    group_size,
    use_fp8: tl.constexpr,
    # number of virtual SM
    NUM_SM: tl.constexpr,
    # tile sizes
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    tile_idx = tl.program_id(0)
    last_problem_end = 0
    gm, gn, gk = group_gemm_sizes
    lda, ldb, ldc = g_lds
    # lda = tl.load(g_lds)
    num_m_tiles = tl.cdiv(gm, BLOCK_SIZE_M)
    num_n_tiles = tl.cdiv(gn, BLOCK_SIZE_N)
    num_tiles = num_m_tiles * num_n_tiles
    # ldb = tl.load(g_lds + 1)
    # ldc = tl.load(g_lds  + 2)
    tl.assume(lda > 0)
    tl.assume(ldb > 0)
    tl.assume(ldc > 0)
   # tl.device_print("dt", tl.float8e4nv)
    tl_dtype = tl.float8e4nv if use_fp8 else tl.float16
    for g in range(group_size):
        # get the gemm size of the current problem
        # tl.device_print("group_gemm_sizes", group_gemm_sizes)
        #tl.device_print("gm", gm)
        #tl.device_print("group_size", group_size)
        #tl.device_print("gn", gn)
        # tl.device_print("gk", gk)
        # print("num_m_tiles", num_m_tiles)
        # print("num_n_tiles", num_n_tiles)
        # print("num_tiles", num_tiles)
        # iterate through the tiles in the current gemm problem
        while (tile_idx >= last_problem_end and tile_idx < last_problem_end + num_tiles):
            # pick up a tile from the current gemm problem
            k = gk
            a_ptr = tl.load(group_a_ptrs + g).to(tl.pointer_type(tl_dtype))
            b_ptr = tl.load(group_b_ptrs + g).to(tl.pointer_type(tl_dtype))
            c_ptr = tl.load(group_c_ptrs + g).to(tl.pointer_type(tl.float16))
            # tl.device_print("a_ptr", a_ptr)
            # figure out tile coordinates
            tile_idx_in_gemm = tile_idx - last_problem_end
            tile_m_idx = tile_idx_in_gemm // num_n_tiles
            tile_n_idx = tile_idx_in_gemm % num_n_tiles

            # do regular gemm here
            offs_am = tile_m_idx * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
            offs_bn = tile_n_idx * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
            offs_k = tl.arange(0, BLOCK_SIZE_K)
            a_ptrs = a_ptr + offs_am[:, None] * lda + offs_k[None, :]
            b_ptrs = b_ptr + offs_k[:, None] * ldb + offs_bn[None, :]
            accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
            for kk in range(0, tl.cdiv(k, BLOCK_SIZE_K)):
                # hint to Triton compiler to do proper loop pipelining
                tl.multiple_of(a_ptrs, [16, 16])
                tl.multiple_of(b_ptrs, [16, 16])
                # assume full tile for now
                a = tl.load(a_ptrs, mask=offs_k[None, :] < gk - kk * BLOCK_SIZE_K, other=0.0) # FIXME there is an error with the load function
                #a = tl.full((BLOCK_SIZE_M, BLOCK_SIZE_K), value=1, dtype=tl.float16)
                #tl.full((BLOCK_WIDTH, K_DIM), value=1, dtype=tl.float32)
                b = tl.load(b_ptrs, mask=offs_k[:, None] < gk - kk * BLOCK_SIZE_K, other=0.0)
                #b = tl.full((BLOCK_SIZE_K, BLOCK_SIZE_N), value=1, dtype=tl.float16)
                accumulator += tl.dot(a, b)
                # tl.device_print("a", a)
                a_ptrs += BLOCK_SIZE_K  # no lda here
                b_ptrs += BLOCK_SIZE_K * ldb
            c = accumulator.to(tl.float16)

            offs_cm = tile_m_idx * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
            offs_cn = tile_n_idx * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
            c_ptrs = c_ptr  + ldc * offs_cm[:, None] + offs_cn[None, :]
            c_mask = (offs_cm[:, None] < gm) & (offs_cn[None, :] < gn)

            # assumes full tile for now
            #tl.store(c_ptrs, tl.full((BLOCK_SIZE_M, BLOCK_SIZE_N), value=1, dtype=tl.float16))
            tl.store(c_ptrs, c,  mask=c_mask) # invalid read 16 bytes

            # go to the next tile by advancing NUM_SM
            tile_idx += NUM_SM

        # get ready to go to the next gemm problem
        last_problem_end = last_problem_end + num_tiles

if __name__ == "__main__":
    K = 5120
    N = 2048 # divde by 8 and multipyl by 2
    niter= 10
    use_fp_8 = False
    num_experts = 128
    if use_fp_8:
        dtype = "fp8"
    else:
        dtype = "fp16"
    configs = get_config(config_file_path = f"/home/ubuntu/vllm/benchmarks/kernels/configs_groups={num_experts}_N={N}_K={5120}_{dtype}.json")
    #configs = get_config(config_file_path = "/home/ubuntu/vllm/benchmarks/kernels/config.json")
    for M in [1,2,8,16,32,64, 128]:
        config = configs[M]
        a = torch.randn((M, K), device=DEVICE, dtype=torch.float16)
        if num_experts > 1:
            b = torch.randn((num_experts, N, K), device=DEVICE, dtype=torch.float16)
        else:
            b = torch.randn((K, N), device=DEVICE, dtype=torch.float16)

        if use_fp_8:
            a = a.to(torch.float8_e4m3fn)
            # b = b.T
            b = b.to(torch.float8_e4m3fn)
        quantiles = [0.5, 0.2, 0.8]

        if not use_fp_8:
            # cublas_ms = triton.testing.do_bench(lambda: torch.matmul(a, b), quantiles=quantiles)
            triton_ms = triton.testing.do_bench(lambda: matmul(a, b, num_experts, config), quantiles=quantiles)
            #print("M", M, "cublasms", cublas_ms)
        else:
            triton_ms = triton.testing.do_bench(lambda: matmul(a, b, num_experts, config), quantiles=quantiles)
        print("M", M, "tritonms", triton_ms)


M 1 tritonms [0.022175999358296394, 0.021695999428629875, 0.023264000192284584]
M 2 tritonms [0.023520000278949738, 0.022975999861955643, 0.0244159996509552]


M 8 tritonms [0.02454400062561035, 0.024064000695943832, 0.025536000728607178]
M 16 tritonms [0.021247999742627144, 0.020800000056624413, 0.022431999444961548]
M 32 tritonms [0.024927999824285507, 0.024224000051617622, 0.025728000327944756]
M 64 tritonms [0.024800000712275505, 0.024480000138282776, 0.025439999997615814]
M 128 tritonms [0.029983999207615852, 0.029664000496268272, 0.030572799965739254]


In [12]:
A = torch.rand((M, K), device=DEVICE, dtype=torch.float16)
B = torch.rand((K, N), device=DEVICE, dtype=torch.float16)
A.stride(0), B.stride(0)

(5120, 2048)

In [13]:
# only launch the kernel, no tensor preparation here to remove all overhead
def triton_perf_fn(a_ptrs, b_ptrs, c_ptrs, sizes, lds, group_size, use_fp8, config):
    grid = lambda META: (META['NUM_SM'],)
    grouped_matmul_kernel[grid](
        a_ptrs,
        b_ptrs,
        c_ptrs,
        sizes,
        lds,
        group_size,
        use_fp8,
        **config,
    )

In [14]:
def group_gemm_fn(group_A, group_B, use_config=False):
    assert len(group_A) == len(group_B)
    group_size = len(group_A)

    A_addrs = []
    B_addrs = []
    C_addrs = []
    g_sizes = []
    g_lds = []
    group_C = []
    for i in range(group_size):
        A = group_A[i]
        B = group_B[i]
        assert A.shape[1] == B.shape[0]
        M, K = A.shape
        K, N = B.shape
        C = torch.empty((M, N), device=DEVICE, dtype=A.dtype)
        group_C.append(C)
        A_addrs.append(A.data_ptr())
        B_addrs.append(B.data_ptr())
        C_addrs.append(C.data_ptr())
        g_sizes += [M, N, K]
        g_lds += [A.stride(0), B.stride(0), C.stride(0)]

    # print(A_addrs, B_addrs, C_addrs)
    # note these are device tensors
    d_a_ptrs = torch.tensor(A_addrs, device=DEVICE)
    d_b_ptrs = torch.tensor(B_addrs, device=DEVICE)
    d_c_ptrs = torch.tensor(C_addrs, device=DEVICE)
    d_g_sizes = torch.tensor(g_sizes, dtype=torch.int32, device=DEVICE)
    d_g_lds = torch.tensor(g_lds, dtype=torch.int32, device=DEVICE)
    # we use a fixed number of CTA, and it's auto-tunable
    config = configs[M]
    config['NUM_SM']= num_sms() 
    config.pop('GROUP_SIZE_M', None)
    # print(config)
    grid = lambda META: (META['NUM_SM'], )
    if use_config:
        grouped_matmul_kernel[grid](
        d_a_ptrs,
        d_b_ptrs,
        d_c_ptrs,
        (M, N, K),
        (K, N, N),
        group_size,
        **config,
        )
    else:
        grouped_matmul_kernel[grid](
        d_a_ptrs,
        d_b_ptrs,
        d_c_ptrs,
        d_g_sizes,
        d_g_lds,
        group_size
    )
    return group_C

In [15]:

def num_sms():
    if is_cuda():
        return torch.cuda.get_device_properties("cuda").multi_processor_count
    return 148

def is_cuda():
    return triton.runtime.driver.active.get_current_target().backend == "cuda"

# run MoE test
def test_moe(M=1, N=2048, K=5192, num_experts=128):
    group_a = []
    group_b = []
    num_activated_experts = min(num_experts, M)
    #print(M, num_activated_experts)
    a = torch.rand((M, K), device=DEVICE, dtype=torch.float16)
    b = torch.rand((num_experts, K, N), device=DEVICE, dtype=torch.float16)
    expert_ids = torch.arange(num_activated_experts, device=a.device, dtype=torch.int32) #.view(-1,1)
    for m in range(num_activated_experts):
        group_a.append(torch.unsqueeze(a[m,:], 0))
        #print(b[expert_ids[m], :, :].shape, torch.unsqueeze(a[m,:], 0).shape)
        group_b.append(b[expert_ids[m], :, :])
        # print(group_a[m].shape, group_b[m].shape)

    quantiles = [0.5, 0.2, 0.8]
    triton_ms = triton.testing.do_bench(lambda: group_gemm_fn(group_a, group_b, use_config=True), quantiles=quantiles)
    #ref_out = [torch.matmul(a, b) for a, b in zip(group_a, group_b)]
    # for i in range(num_activated_experts):
    #     assert torch.allclose(ref_out[i], tri_out[i], atol=1e-2, rtol=1e-2)
    print(triton_ms)

In [16]:
triton_ms

[0.029983999207615852, 0.02969600073993206, 0.030527999624609947]

In [17]:
@triton.jit
def mat_vec_kernel(
    vec_ptr,
    matrix_ptr,
    out_ptr,
    vec_stridex,
    matrix_stridey,
    matrix_stridex,
    out_stridex,
    K,
    BLOCK_SIZE_M: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE_M
    offsets = block_start + tl.arange(0, BLOCK_SIZE_M)
    mask = offsets < K
    # vec_ptr = vec_ptr + vec_stridex * tl.arange(0, BLOCK_SIZE_M)
    #vec_ptr = tl.reshape(vec_ptr, (BLOCK_SIZE_M, 1))

    # out_ptr = out_ptr + out_stridex * tl.arange(0, BLOCK_SIZE_M)
    #out_ptr = tl.reshape(out_ptr, (BLOCK_SIZE_M, 1))

    matrix_x = block_start + matrix_stridex * tl.arange(0, BLOCK_SIZE_M)
    matrix_y =  block_start + matrix_stridey * tl.arange(0, BLOCK_SIZE_M)

    matrix_ptrs = matrix_ptr + (matrix_x[None, :] + matrix_y[:, None])

    mask = tl.arange(0, BLOCK_SIZE_M) < K

    val = tl.load(vec_ptr + offsets, mask=mask).to(tl.float32)
    matrix = tl.load(matrix_ptrs).to(tl.float32)
    #tl.store(out_ptr, tl.sum(val[:, None] * matrix, 0))
    tl.store(out_ptr + offsets, tl.sum(val[:, None] *matrix, 0), mask=mask)

In [18]:
@triton.jit
def matmul_kernel(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        # The stride variables represent how much to increase the ptr by when moving by 1
        # element in a particular dimension. E.g. `stride_am` is how much to increase `a_ptr`
        # by to get the element one row down (A has M rows).
        stride_am, stride_ak,  #
        stride_bk, stride_bn,  #
        stride_cm, stride_cn,
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,  #
        GROUP_SIZE_M: tl.constexpr,  #
        ACTIVATION: tl.constexpr  #
):
    """Kernel for computing the matmul C = A x B.
    A has shape (M, K), B has shape (K, N) and C has shape (M, N)
    """
    # -----------------------------------------------------------
    # Map program ids `pid` to the block of C it should compute.
    # This is done in a grouped ordering to promote L2 data reuse.
    # See above `L2 Cache Optimizations` section for details.
    # Program ID
    pid = tl.program_id(axis=0) 
    # Number of program ids along the M axis
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    # Number of programs ids along the N axis
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    # Number of programs in group
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    # Id of the group this program is in
    group_id = pid // num_pid_in_group
    # Row-id of the first program in the group
    first_pid_m = group_id * GROUP_SIZE_M
    # If `num_pid_m` isn't divisible by `GROUP_SIZE_M`, the last group is smaller
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    # *Within groups*, programs are ordered in a column-major order
    # Row-id of the program in the *launch grid*
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    # Col-id of the program in the *launch grid*
    pid_n = (pid % num_pid_in_group) // group_size_m

    # -----------------------------------------------------------
    # Add some integer bound assumptions.
    # This helps to guide integer analysis in the backend to optimize
    # load/store offset address calculation
    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    # ----------------------------------------------------------
    # Create pointers for the first blocks of A and B.
    # We will advance this pointer as we move in the K direction
    # and accumulate
    # `a_ptrs` is a block of [BLOCK_SIZE_M, BLOCK_SIZE_K] pointers
    # `b_ptrs` is a block of [BLOCK_SIZE_K, BLOCK_SIZE_N] pointers
    # See above `Pointer Arithmetic` section for details
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    # -----------------------------------------------------------
    # Iterate to compute a block of the C matrix.
    # We accumulate into a `[BLOCK_SIZE_M, BLOCK_SIZE_N]` block
    # of fp32 values for higher accuracy.
    # `accumulator` will be converted back to fp16 after the loop.
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        # Load the next block of A and B, generate a mask by checking the K dimension.
        # If it is out of bounds, set it to 0.
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        # We accumulate along the K dimension.
        accumulator = tl.dot(a, b, accumulator)
        # Advance the ptrs to the next K block.
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    # You can fuse arbitrary activation functions here
    # while the accumulator is still in FP32!
    if ACTIVATION == "leaky_relu":
        accumulator = leaky_relu(accumulator)
    c = accumulator.to(tl.float16)

    # -----------------------------------------------------------
    # Write back the block of the output matrix C with masks.
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

In [19]:
def matmul(a, b, activation=""):
    # Check constraints.
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.is_contiguous(), "Matrix A must be contiguous"
    M, K = a.shape
    K, N = b.shape
    configs = get_config(config_file_path = "/home/ubuntu/vllm/benchmarks/kernels/config.json")
    config = configs[M]
    # Allocates output.
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    # 1D launch kernel where each block gets its own program.
    grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
    matmul_kernel[grid](
        a, b, c,  #
        M, N, K,  #
        a.stride(0), a.stride(1),  #
        b.stride(0), b.stride(1),  #
        c.stride(0), c.stride(1),  #
        ACTIVATION=activation,  #
        **config
    )
    return c

In [20]:
@triton.jit
def row_vector_matrix_multiply_kernel(
    x_ptr, # Pointer to the row vector (1 x K)
    A_ptr, # Pointer to the matrix (K, N)
    y_ptr, # Pointer to the output vector (1 x N)
    K, # Number of columns in x and rows in A
    N, # Number of columns in A and in y
    BLOCK_SIZE_K: tl.constexpr, # Block size for K dimension
    BLOCK_SIZE_N: tl.constexpr, # Block size for N dimension
):
    # Get the program ID for the N dimension
    pid_n = tl.program_id(axis=0)

    # Calculate offsets for the K dimension (for loading x and A)
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    offs_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)

    # Initialize accumulator for the output vector block
    acc = tl.zeros((BLOCK_SIZE_N,), dtype=tl.float32)

    # Loop over the K dimenseion (columns of x rows of A)
    for k_start in range(0, K, BLOCK_SIZE_K):
        # Load block of row vector x
        # This will be a 1D vector of BLOCK_SIZE_K
        x_block_ptr = x_ptr + k_start + offs_k
        x_block = tl.load(x_block_ptr, mask  = offs_k < K, other = 0.0)

        # Load block of matrix A
        # This will be a 2D block of size (BLOCK_SIZE_K, BLOCK_SIZE_N)
        # Note the stride for A to access elements efficiently
        A_block_ptr = A_ptr + (k_start + offs_k[:, None]) * N + offs_n[None, :]
        A_block = tl.load(A_block_ptr, mask = (offs_k[:,None] < K) & (offs_n[None, :] < N), other=0.0)

        # Perform block multiplication and accumulation
        acc += tl.sum(x_block[:, None] * A_block, axis=0) # tl.dot(x_block, A_block)

    # Store the accumulated results into the outptu vector y
    # This will be a 1d vector of size BLOCK_SIZE_N
    #y_offs = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    tl.store(y_ptr + offs_n, acc, mask = offs_n < N)

# Wrapper fucntion to launch the kernel
def row_vector_times_matrix(x, A):
    # Ensure inputs are contiguous
    x = x.contiguous()
    A = A.contiguous()

    # Get dimensions
    assert x.shape[0] == 1, "x must be a row vector"
    assert x.shape[1] == A.shape[0], "Inner dimension must match"
    K = x.shape[1]
    N = A.shape[1]

    # Output vector
    y = torch.empty((1,N), device=x.device, dtype=x.dtype)

    # Configure block sizes (these are crucial for performance might need tuning)
    BLOCK_SIZE_K = 32
    BLOCK_SIZE_N = 32

    # Calculate grid dimensions based on N dimension
    grid = (triton.cdiv(N, BLOCK_SIZE_N),)

    # Launch the kernel
    row_vector_matrix_multiply_kernel[grid](
        x, A, y, K, N, BLOCK_SIZE_K, BLOCK_SIZE_N
    )
    return y

In [21]:
def test_moe_mat_vec(M=1, N=2048, K=5120, num_experts=128, use_fp8=False,
                     dtype_fp8 = torch.float8_e4m3fn, is_mat_vec=False):
    group_A = []
    group_B = []
    A_addrs = []
    B_addrs = []
    C_addrs = []
    g_sizes = []
    g_lds = []
    group_C = []
    num_activated_experts = min(num_experts, M)
    A_total = torch.rand((M, K), device=DEVICE, dtype=torch.float16)
    B_total = torch.rand((num_experts, K, N), device=DEVICE, dtype=torch.float16)
    expert_ids = torch.arange(num_activated_experts, device=DEVICE, dtype=torch.int32) #.view(-1,1)
    times = torch.zeros((num_activated_experts, 3))
    for m in range(num_activated_experts):
        A = torch.unsqueeze(A_total[m,:], 0)
        A = A.contiguous()
        M_e = A.shape[0]
        B = B_total[expert_ids[m], :, :]
        B = B.contiguous()
        if use_fp8:
            A = A.to(dtype_fp8)
            # b = b.T
            B = B.to(dtype_fp8)
        C = torch.empty((M_e, N), device=DEVICE, dtype=torch.float16)

       # B_T = B.T.contiguous()
        group_A.append(A)
        group_B.append(B)
        # group_B_T.append(B_T)
        group_C.append(C)

        quantiles = [0.5, 0.2, 0.8]
        if is_mat_vec:
            # configs = get_config(config_file_path = "/home/ubuntu/vllm/benchmarks/kernels/config_vec.json")
            # config = configs[M_e]
            # grid = lambda meta: (triton.cdiv(K, meta['BLOCK_SIZE_M']), )
            # times[m,0], times[m,1], times[m,2] = triton.testing.do_bench(
            # lambda: mat_vec_kernel[grid](A, B, C, A.stride(1), B.stride(0), B.stride(1), C.stride(1),K, **config), quantiles=quantiles)
            BLOCK_SIZE_K = 128
            BLOCK_SIZE_N = 64
            grid = (triton.cdiv(N, BLOCK_SIZE_N),)
            times[m,0], times[m,1], times[m,2] = triton.testing.do_bench(lambda: row_vector_matrix_multiply_kernel[grid](
                        A, B, C, K, N, BLOCK_SIZE_K, BLOCK_SIZE_N), quantiles=quantiles
           )
        else:
            if use_fp8:
                dtype = "fp8"
            else:
                dtype = "fp16"
            config_file_path = f"/home/ubuntu/vllm/benchmarks/kernels/configs_groups=128_N=2048_K=5120_{dtype}.json"
            configs = get_config(config_file_path)
            config=configs[M_e]
            # times[m,0], times[m,1], times[m,2] = triton.testing.do_bench(
            # lambda: matmul(A,B), quantiles=quantiles)
            grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
            times[m,0], times[m,1], times[m,2] = triton.testing.do_bench(lambda: matmul_kernel[grid](
            A, B, C,  #
            M_e, N, K,  #
            A.stride(0), A.stride(1),  #
            B.stride(0), B.stride(1),  #
            C.stride(0), C.stride(1),  #
            ACTIVATION="",
            **config) ,
            quantiles=quantiles)
    # #ref_out = [torch.matmul(a, b) for a, b in zip(group_a, group_b)]
    # # for i in range(num_activated_experts):
    # #     assert torch.allclose(ref_out[i], tri_out[i], atol=1e-2, rtol=1e-2)
    return torch.sum(times, axis=0) #, group_A, group_B, group_C
    #return  group_A, group_B, group_C

In [22]:
group_A, group_B, group_C = test_moe_mat_vec(M=4, is_mat_vec=False, use_fp8=False)

In [23]:
group_C

tensor(0.0802)

In [21]:
for i in range(len(group_A)):
    ref = torch.matmul(group_A[i],group_B[i])
    print(ref)

tensor([[1289., 1291., 1310.,  ..., 1280., 1315., 1297.]], device='cuda:0',
       dtype=torch.float16)
tensor([[1293., 1293., 1285.,  ..., 1313., 1270., 1303.]], device='cuda:0',
       dtype=torch.float16)
tensor([[1295., 1290., 1300.,  ..., 1278., 1265., 1274.]], device='cuda:0',
       dtype=torch.float16)
tensor([[1289., 1318., 1282.,  ..., 1296., 1300., 1265.]], device='cuda:0',
       dtype=torch.float16)


In [12]:
niter = 8
for i in range(niter):
    M = 2**i
    ms, max_ms, min_ms = test_moe_mat_vec(M, is_mat_vec=False, use_fp8=False)
    print("M", M, ms, max_ms, min_ms)

M 1 tensor(0.0195) tensor(0.0192) tensor(0.0198)
M 2 tensor(0.0387) tensor(0.0382) tensor(0.0395)
M 4 tensor(0.0776) tensor(0.0766) tensor(0.0791)
M 8 tensor(0.1552) tensor(0.1532) tensor(0.1581)
M 16 tensor(0.3110) tensor(0.3070) tensor(0.3168)
M 32 tensor(0.7055) tensor(0.6936) tensor(0.7204)
M 64 tensor(1.7057) tensor(1.6899) tensor(1.7237)


RuntimeError: CUDA error: an illegal memory access was encountered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [28]:
def test_moe_perf(M=1, N=2048, K=5120, num_experts=128, use_fp8=False, dtype_fp8 = torch.float8_e4m3fn, is_time=True):
    group_A = []
    group_B = []
    A_addrs = []
    B_addrs = []
    C_addrs = []
    group_C = []
    num_activated_experts = min(num_experts, M)
    A_total = torch.rand((M, K), device=DEVICE, dtype=torch.float16)
    B_total = torch.rand((num_experts, K, N), device=DEVICE, dtype=torch.float16)
    expert_ids = torch.arange(num_activated_experts, device=DEVICE, dtype=torch.int32) #.view(-1,1)
    if use_fp8:
        dtype = "fp8"
    else:
        dtype = "fp16"
    config_file_path = f"/home/ubuntu/vllm/benchmarks/kernels/configs_groups={num_experts}_N={N}_K={K}_fp16_groupgemm.json"
    configs = get_config(config_file_path = config_file_path)
    #onfigs = get_config(config_file_path = f"/home/ubuntu/vllm/benchmarks/kernels/config.json")
    config = configs[M]
    for m in range(num_activated_experts):
        A = torch.unsqueeze(A_total[m,:], 0)
        M_e = A.shape[0]
        B = B_total[expert_ids[m], :, :]
        config = configs[M_e]
        config['NUM_SM']= num_sms() 
        config.pop('GROUP_SIZE_M', None)
        if use_fp8:
            A = A.to(dtype_fp8)
            # b = b.T
            B = B.to(dtype_fp8)
        C = torch.empty((M_e, N), device=DEVICE, dtype=torch.float16)

        group_A.append(A)
        group_B.append(B)
        group_C.append(C)
        A_addrs.append(A.data_ptr())
        B_addrs.append(B.data_ptr())
        C_addrs.append(C.data_ptr())
        # g_sizes += [M_e, N, K]
        # g_lds += [A.stride(0), B.stride(0), C.stride(0)]
        # g_T_lds += [A.stride(0), B_T.stride(0), C.stride(0)]

    d_a_ptrs = torch.tensor(A_addrs, device=DEVICE)
    d_b_ptrs = torch.tensor(B_addrs, device=DEVICE)
    #d_b_t_ptrs = torch.tensor(B_T_addrs, device=DEVICE)
    d_c_ptrs = torch.tensor(C_addrs, device=DEVICE)
    # d_g_sizes = torch.tensor(g_sizes, dtype=torch.int32, device=DEVICE)
    # d_g_lds = torch.tensor(g_lds, dtype=torch.int32, device=DEVICE)
    # d_g_t_lds = torch.tensor(g_T_lds, dtype=torch.int32, device=DEVICE)
    quantiles = [0.5, 0.2, 0.8]
    ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: triton_perf_fn(d_a_ptrs, d_b_ptrs, d_c_ptrs,(M_e, N, K),
                                    (K, N, N), num_activated_experts, use_fp8, config), quantiles=quantiles)
    #ref_out = [torch.matmul(a, b) for a, b in zip(group_a, group_b)]
    # for i in range(num_activated_experts):
    #     assert torch.allclose(ref_out[i], tri_out[i], atol=1e-2, rtol=1e-2)
    if is_time:
        return  ms, min_ms, max_ms
    else:
        return  group_A, group_B, group_C


In [34]:
M= 1

In [37]:
group_A, group_B, group_C = test_moe_perf(M, use_fp8=True, is_time=False)

In [38]:
group_C

[tensor([[1261., 1255., 1252.,  ..., 1254., 1252., 1272.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1295., 1287., 1292.,  ..., 1290., 1305., 1286.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1283., 1275., 1261.,  ..., 1309., 1298., 1271.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1313., 1295., 1294.,  ..., 1302., 1310., 1301.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1284., 1307., 1275.,  ..., 1275., 1275., 1282.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1262., 1257., 1273.,  ..., 1268., 1287., 1286.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1267., 1249., 1254.,  ..., 1245., 1279., 1256.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1283., 1283., 1286.,  ..., 1291., 1289., 1293.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1274., 1233., 1243.,  ..., 1237., 1265., 1260.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1286., 1254., 1261.,  ..., 

In [40]:
for i in range(10):
    M = 2 ** i
    if M == 4:
        continue
    ms, min_ms, max_ms = test_moe_perf(M, use_fp8=True)
    print ("M", M, ms)

M 1 0.01836800016462803
M 2 0.022016000002622604
M 8 0.06275200098752975
M 16 0.11059200018644333
M 32 0.20479999482631683
M 64 0.3898240029811859
M 128 0.7611039876937866


KeyError: 256

In [70]:
group_C

[tensor([[1282., 1252., 1280.,  ..., 1274., 1269., 1280.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1285., 1278., 1300.,  ..., 1281., 1280., 1272.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1307., 1266., 1265.,  ..., 1256., 1288., 1261.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1280., 1260., 1282.,  ..., 1296., 1267., 1271.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1262., 1267., 1280.,  ..., 1245., 1257., 1279.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1274., 1264., 1266.,  ..., 1244., 1237., 1280.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1275., 1302., 1267.,  ..., 1272., 1291., 1280.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1272., 1270., 1285.,  ..., 1247., 1293., 1259.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1264., 1273., 1252.,  ..., 1274., 1276., 1289.]], device='cuda:0',
        dtype=torch.float16),
 tensor([[1283., 1297., 1279.,  ..., 

In [69]:
torch.matmul(group_A[127], group_B[127])

tensor([[1296., 1274., 1266.,  ..., 1267., 1268., 1257.]], device='cuda:0',
       dtype=torch.float16)

In [45]:
# TODO: Add test, grouped GEMM and matvec kernel!
for i in range(len(group_A)):
    ref = torch.matmul(group_A[i], group_B[i])
    assert torch.allclose(group_C[i], ref, atol=1e-2, rtol=1e-2)
    # print(torch.max(abs(group_C[i])- torch.matmul(group_A[i], group_B[i])))
# torch.matmul(group_A[127], group_B[127])

In [17]:
niter = 8
for i in range(niter):
    M = 2**i
    ms, max_ms, min_ms = test_moe_perf(M, use_fp8=False)
    print("M", M, ms, max_ms, min_ms)

CompilationError: at 35:12:
    gm, gn, gk = group_gemm_sizes
    lda, ldb, ldc = g_lds
    # lda = tl.load(g_lds)
    num_m_tiles = tl.cdiv(gm, BLOCK_SIZE_M)
    num_n_tiles = tl.cdiv(gn, BLOCK_SIZE_N)
    num_tiles = num_m_tiles * num_n_tiles
    # ldb = tl.load(g_lds + 1)
    # ldc = tl.load(g_lds  + 2)
    tl.assume(lda > 0)
    tl.assume(ldb > 0)
    tl.assume(ldc > 0)
    dtype = tl.float8e4nv if use_fp8 else tl.float16
            ^
TypeError("cannot convert fp8e4nv of type <class 'triton.language.core.dtype'> to tensor")

In [16]:
group_C

[tensor([[  3.8555,   3.9766,   4.9531,  ..., -60.2812,   2.0117, -38.9062]],
        device='cuda:0', dtype=torch.float16)]